# 1. Collaborative filtering

**Member:** 3 (Collaborative Filtering)  
**Branch:** `m3-collaborative`

## Progress
- [x] Sparse customer-by-article interaction matrix
- [x] Item-to-item collaborative filtering
- [x] ALS with implicit feedback and a small parameter comparison
- [x] Compare Item-to-Item CF and ALS by customer segment
- [x] Analyse limits for light customers and new articles


In [1]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from hm_recsys.data import PROCESSED_DIR, RAW_DIR, new_items
from hm_recsys.baselines import PopularityRecommender
from hm_recsys.collaborative import ALSRecommender, ItemItemCF
from hm_recsys.evaluation import evaluate


## 2. Interaction matrix

Rows represent customers and columns represent articles. A nonzero entry marks a purchase; repeated transactions are collapsed to one interaction. CSR storage keeps this large matrix sparse.


## 3. Item-to-item collaborative filtering

Cosine similarity links articles bought by the same customers. Each article keeps at most 100 neighbours. A customer's purchase history scores other articles; purchased items are excluded and popularity fills any remaining slots. Both models exclude prior purchases, so repeat purchases in the test week cannot count as hits and may lower evaluation scores.


## 4. ALS with implicit feedback

ALS learns latent customer and article factors from positive purchase signals, not ratings. The `implicit` library scales observed interaction confidence by `alpha=40` (a binary purchase therefore has confidence 40; unobserved pairs retain baseline confidence 1). A fixed random seed supports reproducibility.


## 5. ALS parameter comparison

Use the final seven days of `train.parquet` for validation and fit on the earlier training rows. The final test week is not loaded. A fixed sample of 10,000 validation customers keeps the comparison manageable; every setting uses the same customers and all their validation transactions.

Compare 32 and 64 factors with regularization 0.01 and 0.1. Keep 15 iterations and `alpha=40` fixed. Select by MAP@12, with Recall@12 and NDCG@12 as supporting metrics.


### 5.1 Internal validation split

The last week of the training period is held out chronologically. Only the training parquet and the columns needed for collaborative filtering are read.


In [2]:
transactions = pd.read_parquet(
    PROCESSED_DIR / "train.parquet",
    columns=["t_dat", "customer_id", "article_id"],
)
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"])

validation_start = transactions["t_dat"].max().normalize() - pd.Timedelta(days=6)
fit_train = transactions.loc[transactions["t_dat"] < validation_start].copy()
validation = transactions.loc[transactions["t_dat"] >= validation_start].copy()

validation_customer_ids = (
    validation["customer_id"].astype("string").drop_duplicates()
)
sampled_customer_ids = validation_customer_ids.sample(
    n=min(10_000, len(validation_customer_ids)), random_state=42
)
validation_sample = validation.loc[
    validation["customer_id"].astype("string").isin(sampled_customer_ids)
].copy()

print(f"Fit period: {fit_train['t_dat'].min().date()} to {fit_train['t_dat'].max().date()}")
print(f"Validation period: {validation['t_dat'].min().date()} to {validation['t_dat'].max().date()}")
print(f"Fit rows: {len(fit_train):,}; validation rows: {len(validation):,}")
print(f"Validation sample: {len(sampled_customer_ids):,} customers, {len(validation_sample):,} transactions")

Fit period: 2020-06-24 to 2020-09-08
Validation period: 2020-09-09 to 2020-09-15
Fit rows: 3,408,839; validation rows: 255,241
Validation sample: 10,000 customers, 35,031 transactions


### 5.2 Fit and evaluate each setting

The shared `evaluate()` function scores the same validation sample for each model. The output reports the three ranking metrics and fit/evaluation times.


In [3]:
results = []
for factors in (32, 64):
    for regularization in (0.01, 0.1):
        model = ALSRecommender(
            factors=factors, regularization=regularization, iterations=15
        )
        fit_start = time.perf_counter()
        model.fit(fit_train)
        fit_seconds = time.perf_counter() - fit_start

        evaluation_start = time.perf_counter()
        metrics = evaluate(model, validation_sample, k=12, train=fit_train)
        evaluation_seconds = time.perf_counter() - evaluation_start
        results.append({
            "factors": factors,
            "regularization": regularization,
            "iterations": 15,
            "MAP@12": metrics.loc["all", "MAP@12"],
            "Recall@12": metrics.loc["all", "Recall@12"],
            "NDCG@12": metrics.loc["all", "NDCG@12"],
            "fit_seconds": fit_seconds,
            "evaluation_seconds": evaluation_seconds,
        })

results = pd.DataFrame(results).sort_values("MAP@12", ascending=False)
display(results.reset_index(drop=True))

,factors,regularization,iterations,MAP@12,Recall@12,NDCG@12,fit_seconds,evaluation_seconds
0,64,0.01,15,0.007511,0.019517,0.012753,23.929886,2.126889
1,64,0.10,15,0.007431,0.019171,0.012579,24.025935,2.126947
2,32,0.01,15,0.006466,0.017644,0.011295,11.427309,2.610749
3,32,0.10,15,0.006436,0.017338,0.011203,11.340162,2.596145


### 5.3 Parameter choice

The 64-factor model with regularization 0.01 had the highest MAP@12 (0.007511), so it was selected. It also led on Recall@12 (0.019517) and NDCG@12 (0.012753). The 64-factor settings took about twice as long to fit as the 32-factor settings, but improved all three validation metrics.


## 6. Refit the selected model

After selecting by validation MAP@12, refit that configuration on all rows in the original training period, including the internal validation week. The final test period remains unused.


In [4]:
best = results.iloc[0]
selected_model = ALSRecommender(
    factors=int(best["factors"]),
    regularization=float(best["regularization"]),
    iterations=15,
)
refit_start = time.perf_counter()
selected_model.fit(transactions)
refit_seconds = time.perf_counter() - refit_start

print(
    f"Selected factors={selected_model.factors}, "
    f"regularization={selected_model.regularization}, "
    f"iterations={selected_model.iterations}; "
    f"full-training refit took {refit_seconds:.1f} seconds."
)

Selected factors=64, regularization=0.01, iterations=15; full-training refit took 25.6 seconds.


## 7. Final model comparison

Fit all three models on the official training split and evaluate them on the full held-out week (16-22 September 2020). Passing training data to `evaluate()` lets it derive the team's customer segments and training popularity; article metadata supplies catalogue coverage and diversity. Test transactions are used only as evaluation targets.


In [5]:
official_train = transactions
official_test = pd.read_parquet(
    PROCESSED_DIR / "test.parquet",
    columns=["customer_id", "article_id"],
)
article_info = pd.read_csv(
    RAW_DIR / "articles.csv",
    usecols=["article_id", "product_group_name"],
)

models = [
    ("ItemItemCF (100 neighbors)", ItemItemCF(n_neighbors=100), None),
    ("ALS (64 factors, reg=0.01)", selected_model, refit_seconds),
    ("Popularity (1 week)", PopularityRecommender(weeks=1), None),
]
model_scores = {}
comparison_rows = []
new_test_articles = len(new_items(official_train, official_test))

for name, model, fit_seconds in models:
    if fit_seconds is None:
        fit_start = time.perf_counter()
        model.fit(official_train)
        fit_seconds = time.perf_counter() - fit_start

    evaluation_start = time.perf_counter()
    scores = evaluate(
        model, official_test, k=12,
        train=official_train, articles=article_info,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start
    model_scores[name] = scores
    overall = scores.loc["all"]
    comparison_rows.append({
        "Model": name,
        "Fit (s)": fit_seconds,
        "Evaluation (s)": evaluation_seconds,
        **{metric: overall[metric] for metric in [
            "MAP@12", "Recall@12", "NDCG@12", "HitRate@12",
            "coverage", "gini", "diversity", "novelty",
        ]},
    })

overall_results = pd.DataFrame(comparison_rows)
print(
    f"Full test customers: {official_test['customer_id'].nunique():,}; "
    f"test-only articles: {new_test_articles:,}"
)
display(overall_results)


Full test customers: 68,984; test-only articles: 922


,Model,Fit (s),Evaluation (s),MAP@12,Recall@12,NDCG@12,HitRate@12,coverage,gini,diversity,novelty
0,ItemItemCF (100 neighbors),7.361685,14.767022,0.007034,0.018184,0.012008,0.042778,0.320422,0.780891,0.346563,14.193131
1,"ALS (64 factors, reg=0.01)",25.634549,13.246366,0.007511,0.020530,0.013221,0.049301,0.032963,0.777954,0.272492,11.192617
2,Popularity (1 week),0.271310,3.347450,0.008984,0.025596,0.015950,0.065783,0.000114,0.000000,0.250000,11.382050


## 8. Performance by customer segment

The shared evaluator uses the project definitions: new customers have no training purchases, light customers have 1-4, and regular customers have 5 or more. The table shows the four ranking metrics for the same full test customer set.


In [6]:
segment_rows = []
for model_name, scores in model_scores.items():
    for segment in ("new", "light", "regular"):
        row = scores.loc[segment]
        segment_rows.append({
            "Model": model_name,
            "Segment": segment,
            "Customers": int(row["n_customers"]),
            "MAP@12": row["MAP@12"],
            "Recall@12": row["Recall@12"],
            "NDCG@12": row["NDCG@12"],
            "HitRate@12": row["HitRate@12"],
        })
segment_results = pd.DataFrame(segment_rows)
display(segment_results)


,Model,Segment,Customers,MAP@12,Recall@12,NDCG@12,HitRate@12
0,ItemItemCF (100 neighbors),new,19765,0.004541,0.014699,0.009164,0.041892
1,ItemItemCF (100 neighbors),light,16051,0.010123,0.021787,0.014950,0.038938
2,ItemItemCF (100 neighbors),regular,33168,0.007026,0.018518,0.012279,0.045164
3,"ALS (64 factors, reg=0.01)",new,19765,0.004541,0.014699,0.009164,0.041892
4,"ALS (64 factors, reg=0.01)",light,16051,0.010823,0.025602,0.016759,0.047910
5,"ALS (64 factors, reg=0.01)",regular,33168,0.007678,0.021549,0.013927,0.054390
6,Popularity (1 week),new,19765,0.008016,0.025804,0.015036,0.065621
7,Popularity (1 week),light,16051,0.009647,0.027684,0.016614,0.063049
8,Popularity (1 week),regular,33168,0.009240,0.024462,0.016174,0.067203


## 9. Limitations and hybrid implications

- **New customers:** ItemItemCF and ALS have no purchase history to score, so both return their all-time popularity fallback. Their new-segment results are identical and below the one-week popularity model on all four ranking metrics.
- **Light customers:** ALS has higher MAP@12 and NDCG@12 than the trending baseline, but lower Recall@12 and HitRate@12. ItemItemCF has a small MAP@12 advantage, while trailing on the other ranking metrics. Results are mixed rather than a clear win.
- **Regular customers:** One-week popularity leads both CF models on MAP@12, Recall@12, NDCG@12, and HitRate@12 in this test week. CF should not be assumed to improve results just because a customer has more history.
- **New articles:** 922 articles in the test week were absent from training. Neither CF model has a learned item representation for those IDs, and a sales-based popularity model cannot rank them either. A content model can use article attributes to give new products scores.
- **Hybrid:** Use recent popularity for customers without history and content-based recommendations for unseen products. CF can remain one signal for customers with history, but the comparison suggests evaluating any blend against the strong trending baseline, especially for regular customers.

ItemItemCF has higher catalogue coverage, diversity, and novelty overall than the other two models. Popularity's zero Gini comes from showing the same 12 items to everyone; its catalogue coverage is correspondingly very low.


## 10. Conclusion

On the full held-out week, one-week popularity had the strongest overall ranking metrics. Item-to-item CF offered broader coverage and more diverse, less popular recommendations, while ALS was mixed by segment. These results support testing CF and content as hybrid signals, not replacing the trending baseline. The evidence covers one test week, so it should not be treated as a general performance guarantee.
